# 🏔️ RuteStrip — Sistem Rekomendasi Jalur Pendakian Gunung di Jawa
## Content-Based Filtering berbasis SBERT dan Cosine Similarity (versi web rutestrip.ai.id)

Notebook ini menjalankan **kode yang sama dengan website** (`python/processor.py` dan `python/search_scoring.py` dari repository `dasrams31/rutestrip-app`), sehingga hasilnya konsisten dengan aplikasi yang berjalan.

**Alur sistem**
1. **Data GPX** → statistik jalur (jarak 3D, total tanjakan, estimasi waktu Naismith, kemiringan rata-rata, elevasi min/maks).
2. **Tingkat kesulitan** dari profil jalur (Shenandoah difficulty rating + faktor kecuraman & ketinggian).
3. **Narasi otomatis** dari statistik → teks konten tiap jalur.
4. **Preprocessing** (regex cleaning, case folding, stopword removal selektif, tanpa stemming).
5. **SBERT** `paraphrase-multilingual-MiniLM-L12-v2` (384 dimensi) → embedding dokumen jalur dan per kalimat aspek.
6. **Cosine similarity** antara query dan jalur → **skor hybrid** (cosine + kecocokan atribut GPX + kecocokan nama) → Top-N.
7. **Evaluasi** Precision@5 dan MRR.

In [1]:
#@title 1️⃣ Setup: install library & ambil kode + data dari repository
import os, sys, subprocess

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gpxpy', 'sentence-transformers', 'scikit-learn', 'pandas'], check=True)

REPO_DIR = os.environ.get('RUTESTRIP_REPO_DIR', 'rutestrip-app')
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/dasrams31/rutestrip-app.git', REPO_DIR], check=True)

PY_DIR = os.path.join(REPO_DIR, 'python')
GPX_DIR = os.path.join(REPO_DIR, 'webchat-api', 'gpx_db')
if PY_DIR not in sys.path:
    sys.path.insert(0, PY_DIR)

import warnings; warnings.filterwarnings('ignore')
import inspect, re, hashlib, json, math
import numpy as np
import pandas as pd

import processor        # GPX processing, Naismith, difficulty, narrative, preprocessing, SBERT
import search_scoring   # hybrid scoring yang dipakai website

print('Repository :', os.path.abspath(REPO_DIR))
print('File GPX   :', len([f for f in os.listdir(GPX_DIR) if f.endswith('.gpx')]))

Repository : /content/rutestrip-app
File GPX   : 98


## 2️⃣ Preprocessing teks
Sama dengan paper: hapus URL, karakter non-alfanumerik, angka berdiri sendiri, normalisasi spasi, case folding, lalu **stopword removal selektif** (kata negasi dan kata sifat jalur seperti *landai, curam, pemula* tetap dipertahankan). Tidak memakai stemming karena SBERT peka konteks.

In [2]:
#@title Kode `preprocess_text` (python/processor.py)
print(inspect.getsource(processor.preprocess_text))
print('Stopwords      :', len(processor.STOPWORDS_ID))
print('Preserve words :', sorted(processor.PRESERVE_WORDS))
contoh = 'Ini adalah jalur yang sangat indah dan tidak terlalu curam, cocok untuk pemula. Info: https://example.com'
print('\nAsli  :', contoh)
print('Hasil :', processor.preprocess_text(contoh))

def preprocess_text(text: str, remove_stopwords: bool = True) -> str:
    """
    Preprocessing teks sesuai paper:
    1. Data Cleaning - Regex untuk hapus karakter non-ASCII, URL, whitespace berlebih
    2. Case Folding - Lowercase
    3. Stopword Removal - Selektif (pertahankan negasi dan kata sifat krusial)
    4. NO STEMMING - Karena SBERT sensitif terhadap konteks
    """
    if not text:
        return ""
    
    # 1. Data Cleaning
    # Hapus URL
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # Hapus karakter non-ASCII kecuali huruf Indonesia
    text = re.sub(r'[^\w\s\-]', ' ', text)
    # Hapus angka yang berdiri sendiri (pertahankan angka dalam konteks)
    text = re.sub(r'\b\d+\b', '', text)
    # Hapus whitespace berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 2. Case Folding
    text = text.lower()
    
    # 3. Stopword Removal (selektif)
    if remove_stopwords:
        words = text.split()
        filtered_words = []
        for word in word

## 3️⃣ Statistik GPX, Naismith, dan tingkat kesulitan
- **Jarak**: `gpx.length_3d()` (km). Elevasi dihaluskan dulu (`gpx.smooth(vertical=True)`) untuk mengurangi noise GPS.
- **Total tanjakan**: jumlah selisih elevasi positif antar titik.
- **Naismith**: `jarak/5 + tanjakan/600` (jam).
- **Grade rata-rata**: `tanjakan / (jarak × 1000) × 100%`.
- **Tingkat kesulitan** (bukan hanya grade): skor = √(tanjakan[ft] × 2 × jarak[mil]) — *Shenandoah National Park hiking difficulty rating* — dikali 1,15 bila grade rata-rata ≥ 25% dan 1,15 bila puncak ≥ 3.000 m.
  Mudah < 75 ≤ Sedang < 200 ≤ Sulit < 460 ≤ Sangat Sulit; tanjakan ≥ 1.200 m minimal **Sulit**.
  Alasannya: grade saja membuat jalur 40 km dengan rata-rata 6% (Semeru) tampak "sedang", sedangkan bukit 1 km dengan 20% tampak "sangat sulit".

In [3]:
#@title Kode statistik, kesulitan, dan narasi (python/processor.py)
for fn in (processor.calculate_statistics, processor.difficulty_score, processor.classify_difficulty, processor.generate_narrative):
    print(inspect.getsource(fn))

def calculate_statistics(gpx, points):
    """Calculate hiking statistics from GPX data"""
    
    # Jika data waktu kosong, gunakan length_3d()
    has_time = any(p['time'] is not None for p in points)
    
    if has_time:
        distance_m = gpx.length_3d()
    else:
        distance_m = gpx.length_3d()
    
    distance_km = distance_m / 1000.0
    
    # Calculate elevation gain
    elevation_gain = 0
    for i in range(1, len(points)):
        ele_diff = points[i]['ele'] - points[i-1]['ele']
        if ele_diff > 0:
            elevation_gain += ele_diff
    
    # Rumus Naismith: (Jarak_KM / 5) + (Gain_Meter / 600)
    naismith_duration = (distance_km / 5) + (elevation_gain / 600)
    
    # Rumus Grade (%): (Gain / (Jarak_KM * 1000)) * 100
    if distance_km > 0:
        average_grade = (elevation_gain / (distance_km * 1000)) * 100
    else:
        average_grade = 0
    
    # Elevation range
    elevations = [p['ele'] for p in points if p['ele'] > 0]
    min_elevation = min

## 4️⃣ Membangun dataset dari file GPX
Filter kualitas yang sama dengan perintah web `php artisan routes:import-gpx-dir`:
- lewati file duplikat (hash sama) dan jalur yang sama (nama sama, atau statistik nyaris sama pada gunung yang sama);
- lewati file gabungan beberapa jalur ("semua jalur", "4 jalur");
- lewati GPX tanpa elevasi atau > 60 km (ikut merekam perjalanan kendaraan);
- lewati elevasi tidak masuk akal: puncak > 3.800 m (salah satuan), titik terendah < 20% puncak (merekam perjalanan dari dataran rendah), tanjakan > 2× rentang elevasi atau grade > 35% (noise).

File berawalan `Mt. ` adalah dataset awal paper: diproses lebih dulu dan, seperti di web, tidak melewati filter kualitas (filter hanya untuk file tambahan).

In [4]:
#@title Proses semua GPX → dataset jalur
NAME_STOP = r'\b(mt|mount|gunung|gurung|via|jalur|to|bc|basecamp|puncak)\b'

def name_key(name):
    words = sorted(set(re.findall(r'[a-z]+', re.sub(NAME_STOP, ' ', name.lower()))))
    return ' '.join(words)

def pretty_name(raw):
    if raw.startswith('Mt. '):
        return raw                                   # nama asli dataset paper
    n = raw.replace('_', ' ')
    n = re.sub(r'\s+\d{1,2}\s+[a-z]{3}\s+\d{4}$', '', n, flags=re.I)
    n = re.sub(r'\s*\(\d+\)$', '', n)
    n = re.sub(r'\s+-\s+', ' via ', n, count=1)
    n = ' '.join(w[:1].upper() + w[1:] for w in n.split())
    n = re.sub(r'\bvia\b', 'via', n, flags=re.I)
    return n if n.startswith(('Gunung ', 'Bukit ', 'Puncak ')) else 'Gunung ' + n

files = sorted(f for f in os.listdir(GPX_DIR) if f.endswith('.gpx'))
files = [f for f in files if f.startswith('Mt. ')] + [f for f in files if not f.startswith('Mt. ')]

routes, skipped, hashes = [], [], set()
for f in files:
    path = os.path.join(GPX_DIR, f)
    h = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    if h in hashes:
        skipped.append((f, 'file sama')); continue
    if re.search(r'semua\s*jalur|\d\s*jalur', f, re.I):
        skipped.append((f, 'gabungan beberapa jalur')); continue
    r = processor.mode_ingest(path)
    if not r.get('success'):
        skipped.append((f, 'gagal diproses')); continue
    km, gain, grade = r['distance_km'], r['elevation_gain_m'], r['average_grade_pct']
    m = re.search(r'dari (\d+)m\s+hingga (\d+)m', r['narrative_text'])
    lo, hi = (int(m.group(1)), int(m.group(2))) if m else (0, 0)
    original = f.startswith('Mt. ')   # dataset paper: tidak difilter (sama seperti di web)
    if not original and (gain <= 0 or km <= 0 or km > 60):
        skipped.append((f, f'statistik tidak valid ({km} km, {gain} m)')); continue
    if not original and (hi > 3800 or (hi > 1000 and lo < 0.2 * hi)):
        skipped.append((f, f'elevasi tidak masuk akal {lo}-{hi} m')); continue
    if not original and (grade > 35 or (hi - lo > 0 and gain > 2 * (hi - lo))):
        skipped.append((f, f'elevasi noise ({gain} m tanjakan, rentang {hi - lo} m)')); continue
    name = pretty_name(f[:-4]); key = name_key(name)
    dup = next((x for x in routes if x['key'] == key or (set(x['key'].split()) & set(key.split())
                and abs(x['distance_km'] - km) <= 0.03 * max(x['distance_km'], km)
                and abs(x['elevation_gain_m'] - gain) <= 0.05 * max(x['elevation_gain_m'], gain, 1))), None)
    if dup:
        skipped.append((f, 'jalur sama dengan ' + dup['name'])); continue
    hashes.add(h)
    routes.append({'id': len(routes) + 1, 'name': name, 'key': key, 'file': f,
                   'distance_km': km, 'elevation_gain_m': gain, 'naismith_duration_hour': r['naismith_duration_hour'],
                   'average_grade_pct': grade, 'min_elevation': lo, 'max_elevation': hi,
                   'difficulty': processor.classify_difficulty(km, gain, grade, hi),
                   'difficulty_score': round(processor.difficulty_score(km, gain, grade, hi)),
                   'narrative_text': r['narrative_text']})

print(f'Jalur dipakai : {len(routes)}')
print(f'Dilewati      : {len(skipped)}')
pd.DataFrame(skipped, columns=['file', 'alasan']).groupby('alasan').size().sort_values(ascending=False).head(15)

Jalur dipakai : 80
Dilewati      : 18


alasan
file sama                                          3
gabungan beberapa jalur                            3
elevasi tidak masuk akal 4218-10139 m              3
elevasi noise (3624 m tanjakan, rentang 1521 m)    1
elevasi tidak masuk akal 29-2066 m                 1
elevasi tidak masuk akal 134-2843 m                1
elevasi noise (4485 m tanjakan, rentang 1790 m)    1
jalur sama dengan Mt. Argopuro Baderan to Bremi    1
statistik tidak valid (114.51 km, 3813 m)          1
statistik tidak valid (66.81 km, 17079 m)          1
statistik tidak valid (76.05 km, 577 m)            1
statistik tidak valid (8.91 km, 0 m)               1
dtype: int64

In [5]:
#@title Dataset jalur
df = pd.DataFrame([{k: r[k] for k in ('id', 'name', 'distance_km', 'elevation_gain_m', 'naismith_duration_hour',
                                      'average_grade_pct', 'max_elevation', 'difficulty_score', 'difficulty')} for r in routes])
df.columns = ['ID', 'Nama', 'Jarak (km)', 'Tanjakan (m)', 'Naismith (jam)', 'Grade (%)', 'Puncak (m)', 'Skor kesulitan', 'Kesulitan']
display(df.sort_values('Skor kesulitan'))
print(df['Kesulitan'].value_counts().reindex(['mudah', 'sedang', 'sulit', 'sangat sulit']))

,ID,Nama,Jarak (km),Tanjakan (m),Naismith (jam),Grade (%),Puncak (m),Skor kesulitan,Kesulitan
45,46,Gunung Geulis Manggahang via Arjasari,1.22,204,0.58,16.76,1127,32,mudah
57,58,Gunung Patuha via Kawah Putih,1.23,241,0.65,19.60,2425,35,mudah
26,27,Bukit Sikendil via Sibajag,1.24,255,0.68,20.58,1889,36,mudah
55,56,Gunung Palasari via Palintang,1.11,305,0.73,27.63,1862,43,mudah
36,37,Gunung Bongkok,1.10,326,0.76,29.69,937,44,mudah
...,...,...,...,...,...,...,...,...,...
22,23,Mt. Welirang via Tretes,24.31,2478,8.99,10.19,3130,570,sangat sulit
6,7,Mt. Lawu via Cemoro Sewu,26.88,2785,10.02,10.36,3205,635,sangat sulit
13,14,Mt. Sindoro via Kledung,28.22,3106,10.82,11.01,3110,687,sangat sulit
12,13,Mt. Semeru,39.80,2356,11.89,5.92,3623,711,sangat sulit


Kesulitan
mudah            7
sedang          21
sulit           38
sangat sulit    14
Name: count, dtype: int64


## 5️⃣ Representasi dokumen untuk SBERT
- **Dokumen jalur** = `"Jalur pendakian {nama}. {narasi}"` → satu embedding (nama ikut masuk sehingga query "merbabu" bisa cocok secara semantik).
- **Aspek** = setiap kalimat narasi (jarak, waktu, kemiringan, elevasi, kesulitan, kecocokan) + kalimat nama → satu embedding per aspek, agar query pendek seperti "jalur landai" dibandingkan dengan aspek yang paling relevan.
- Query diproses dengan `preprocess_text` yang sama seperti paper.

In [6]:
#@title Load SBERT & buat embedding dokumen + aspek
model = processor.get_model()
print('Dimensi embedding:', model.get_sentence_embedding_dimension())
print(inspect.getsource(search_scoring.build_document))

for r in routes:
    doc = search_scoring.build_document(r)
    vecs = model.encode([doc['text']] + doc['aspects'])
    r['embedding'] = vecs[0].tolist()
    r['aspect_embeddings'] = vecs[1:].tolist()

contoh = search_scoring.build_document(routes[0])
print('Contoh dokumen:', contoh['text'][:200], '...')
print('Jumlah aspek  :', len(contoh['aspects']))

Dimensi embedding: 384
def build_document(route: dict) -> dict:
    """Return the text to embed for a route and its aspect sentences."""
    name = clean_route_name(route.get("name"))
    narrative = (route.get("narrative_text") or "").strip()
    description = (route.get("description") or "").strip()

    text = f"Jalur pendakian {name}. {narrative}"
    if description:
        text += f" {description}"

    aspects = [f"Jalur pendakian {name}."] + split_sentences(narrative) + split_sentences(description)[:6]
    return {"text": text, "aspects": aspects}



Contoh dokumen: Jalur pendakian Gunung Argopuro Baderan to Bremi. Jalur pendakian dengan jarak panjang sekitar 45.1 kilometer. Estimasi waktu tempuh lama selama 14.96 jam menggunakan rumus Naismith. Karakteristik jal ...
Jumlah aspek  : 7


## 6️⃣ Skor relevansi (hybrid)
1. `semantik = 0,4 × cos(Q, dokumen) + 0,6 × max cos(Q, aspek)`, dengan `cos(Q, D) = (Q · D) / (‖Q‖ × ‖D‖)`.
2. Jika query memuat kata intent, skor dicampur dengan kecocokan atribut GPX (bobot 0,45):
   *landai* → kemiringan rendah (0,4 grade + 0,3 tanjakan + 0,3 durasi); *pemula/mudah/santai* → sedikit tanjakan & singkat (0,2 / 0,4 / 0,4);
   *curam/sulit/menantang* → kebalikannya; *singkat/tektok* dan *panjang/ekspedisi* → durasi. Negasi dibaca ("tidak terlalu curam" → landai).
3. Jika query menyebut nama gunung/basecamp, skor dicampur dengan proporsi nama yang cocok (bobot 0,35).

Atribut tidak diberikan pada GPX yang statistiknya tidak valid.

In [7]:
#@title Kode hybrid scoring (python/search_scoring.py)
print(inspect.getsource(search_scoring.detect_intents))
print(inspect.getsource(search_scoring.attribute_score))
print(inspect.getsource(search_scoring.hybrid_score))

def search(query, top_n=5):
    q = model.encode(search_scoring.query_text(query)).tolist()
    scored = [(r, search_scoring.hybrid_score(query, q, r['embedding'], r['aspect_embeddings'], r, routes)['score']) for r in routes]
    return sorted(scored, key=lambda x: -x[1])[:top_n]

def detect_intents(query: str) -> set:
    """Intent words, with a negation in the previous two words flipping the intent
    ("tidak terlalu curam" -> gentle)."""
    words = re.findall(r"[a-z]+", (query or "").lower())
    intents = set()
    for i, word in enumerate(words):
        for intent, vocab in INTENT_WORDS.items():
            if word in vocab:
                negated = any(w in NEGATIONS for w in words[max(0, i - 2):i])
                intents.add(OPPOSITE[intent] if negated else intent)
    if "hard" in intents and intents & {"gentle", "easy"}:
        intents -= {"gentle", "easy", "hard"}
    if "short" in intents and "long" in intents:
        intents -= {"short", "long"}
    return intents

def attribute_score(intents: set, route: dict, all_routes: list) -> float:
    """How well the route's GPX statistics fit the intent, as a 0..1 percentile mix.

    "gentle" (landai) is not only a low average grade: a 40 km route can average
    6% and still be a multi-day climb. It

In [8]:
#@title 7️⃣ Coba pencarian (ubah query di kolom kanan)
QUERY = "jalur landai untuk pemula"  #@param {type:"string"}

print(f"Query: {QUERY!r}\n")
for i, (r, s) in enumerate(search(QUERY), 1):
    print(f"{i}. {r['name']:<42} skor {s:.4f} | {r['distance_km']} km, {r['elevation_gain_m']} m, "
          f"{r['naismith_duration_hour']} jam, grade {r['average_grade_pct']}%, {r['difficulty']}")

Query: 'jalur landai untuk pemula'

1. Gunung Ijen                                skor 0.6657 | 10.2 km, 688 m, 3.19 jam, grade 6.75%, sedang
2. Gunung Anak Krakatau                       skor 0.6336 | 3.85 km, 469 m, 1.55 jam, grade 12.18%, sedang
3. Gunung Tangkuban Parahu                    skor 0.6257 | 10.84 km, 716 m, 3.36 jam, grade 6.61%, sedang
4. Gunung Bukittunggul                        skor 0.6095 | 5.94 km, 769 m, 2.47 jam, grade 12.96%, sedang
5. Gunung Ungaran via Mawar                   skor 0.6087 | 6.85 km, 784 m, 2.68 jam, grade 11.46%, sedang


In [9]:
#@title Contoh hasil beberapa query (Top-3)
rows = []
for q in ['jalur mudah untuk pemula', 'jalur landai', 'trek menantang elevasi tinggi', 'pendakian singkat 2-3 jam',
          'pendakian panjang beberapa hari', 'jalur pemula merbabu', 'gede pangrango', 'kawah ijen']:
    rows.append({'Query': q, 'Top-3 (skor)': ' | '.join(f"{r['name']} ({s:.2f})" for r, s in search(q, 3))})
pd.set_option('display.max_colwidth', None)
display(pd.DataFrame(rows))

,Query,Top-3 (skor)
0,jalur mudah untuk pemula,Gunung Geulis Manggahang via Arjasari (0.71) | Gunung Patuha via Kawah Putih (0.70) | Bukit Sikendil via Sibajag (0.69)
1,jalur landai,Gunung Ijen (0.67) | Gunung Tangkuban Parahu (0.65) | Gunung Anak Krakatau (0.62)
2,trek menantang elevasi tinggi,Gunung Ciremai via Linggajati (0.76) | Mt. Slamet via Dipajaya (0.75) | Gunung Slamet via Baturaden Lama (0.72)
3,pendakian singkat 2-3 jam,Gunung Geulis Manggahang via Arjasari (0.83) | Gunung Palasari via Palintang (0.82) | Gunung Patuha via Kawah Putih (0.82)
4,pendakian panjang beberapa hari,Mt. Argopuro Baderan to Bremi (0.83) | Mt. Semeru (0.81) | Mt. Lawu via Cemoro Sewu (0.80)
5,jalur pemula merbabu,Gunung Merbabu via Wekas (0.64) | Mt. Merbabu via Selo (0.63) | Mt. Merbabu via Thekelan (0.63)
6,gede pangrango,Gunung Gede Pangrango via Cibodas (0.51) | Gunung Sagara (0.19) | Gunung Palasari via Palintang (0.19)
7,kawah ijen,Gunung Patuha via Kawah Putih (0.42) | Gunung Ijen (0.39) | Gunung Geulis Manggahang via Arjasari (0.25)


## 8️⃣ Evaluasi
Label relevansi diturunkan otomatis dari statistik GPX (`python/evaluate_search.py`), sehingga dapat direproduksi:
- query *landai* / *tidak terlalu curam* → grade < 15% dan total tanjakan di sepertiga terendah;
- query *pemula/mudah/santai* → total tanjakan di sepertiga terendah dan grade < 30%;
- query *curam/sulit* → grade di sepertiga tertinggi; *singkat* / *panjang* → durasi Naismith sepertiga terendah / tertinggi;
- query nama gunung → nama jalur mengandung nama gunung tersebut.

Metrik: **Precision@5** (proporsi jalur relevan di 5 teratas) dan **MRR** (rata-rata 1/peringkat jalur relevan pertama).

> **Catatan metodologi:** label relevansi dan boost atribut sama-sama diturunkan dari statistik GPX, sehingga angka ini mengukur *konsistensi ranking dengan definisi intent*, bukan penilaian manusia. Untuk naskah, lengkapi dengan label relevansi manual (misalnya penilaian pendaki/pakar) agar evaluasinya independen.

In [10]:
#@title Precision@5 & MRR
import evaluate_search as ev

res = []
for q, label in ev.QUERIES:
    rel = ev.relevant_ids(routes, label)
    ranked = [(r['id'], s) for r, s in search(q, top_n=len(routes))]
    p5, rr = ev.metrics(ranked, rel)
    res.append({'Query': q, 'Label': label, 'Jalur relevan': len(rel), 'P@5': p5, 'RR': rr, 'Skor top-1': round(ranked[0][1], 4)})
ev_df = pd.DataFrame(res)
display(ev_df)
summary = {'Precision@5': ev_df['P@5'].mean(), 'MRR': ev_df['RR'].mean(), 'Skor top-1 rata-rata': ev_df['Skor top-1'].mean()}
display(pd.Series(summary).round(3).to_frame('Nilai'))

,Query,Label,Jalur relevan,P@5,RR,Skor top-1
0,jalur landai,gentle,8,1.0,1.0,0.6738
1,jalur untuk pemula,easy,23,1.0,1.0,0.6988
2,pendakian santai dan mudah,easy,23,1.0,1.0,0.7523
3,jalur ramah pemula tidak terlalu curam,gentle,8,1.0,1.0,0.6507
4,jalur curam dan menantang,hard,27,1.0,1.0,0.6926
5,jalur sulit untuk pendaki berpengalaman,hard,27,1.0,1.0,0.7917
6,pendakian ekstrem,hard,27,1.0,1.0,0.7287
7,pendakian singkat,short,27,1.0,1.0,0.8240
8,jalur pendek bisa tektok,short,27,1.0,1.0,0.7714
9,jalur panjang untuk ekspedisi,long,27,1.0,1.0,0.8246


,Nilai
Precision@5,0.988
MRR,1.000
Skor top-1 rata-rata,0.710


In [11]:
#@title 9️⃣ Ringkasan (dihitung dari hasil di atas)
print(f"Dataset    : {len(routes)} jalur dari {len(files)} file GPX ({len(skipped)} dilewati karena duplikat/data rusak)")
print(f"Kesulitan  : " + ', '.join(f"{k} {v}" for k, v in df['Kesulitan'].value_counts().items()))
print(f"Query uji  : {len(ev_df)}")
print(f"Precision@5: {summary['Precision@5']:.3f}")
print(f"MRR        : {summary['MRR']:.3f}")
print(f"Skor top-1 : {summary['Skor top-1 rata-rata']:.3f}")

Dataset    : 80 jalur dari 98 file GPX (18 dilewati karena duplikat/data rusak)
Kesulitan  : sulit 38, sedang 21, sangat sulit 14, mudah 7
Query uji  : 17
Precision@5: 0.988
MRR        : 1.000
Skor top-1 : 0.710


In [12]:
#@title 🔟 Export dataset ke CSV
out = df.copy()
out['Narasi'] = [r['narrative_text'] for r in routes]
out.to_csv('rutestrip_routes.csv', index=False)
print('Tersimpan: rutestrip_routes.csv')
if IN_COLAB:
    from google.colab import files
    files.download('rutestrip_routes.csv')

Tersimpan: rutestrip_routes.csv
